# MeChess 14/14: the confirmation split (CPU notebook)

An adapter around `chessme humanlike-confirm` and `chessme humanlike-scaling` (workstream W1 of the strengthening plan). The frozen Maia-3 models and the adapters trained earlier (**not refitted**) are evaluated on a **new test set** built from other byte windows of the same Lichess month (games the adapters never saw) and judged by Stockfish with the same node budget: the model-minus-human blunder-rate gap by clock regime, the likelihood change of each adapter, and the change of the gap with model size. Nothing is tuned here; the weights `LAMBDAS` and the comparisons are fixed in the plan before this run.

**Before you run:** add as inputs the outputs of: notebook 6 run for the training set (`TRAIN`), notebook 7 for it, notebook 8 for it per size, the adapter runs of notebook 9 (`ADAPTERS`), and notebooks 6, 7 and 8 run for the new windows (`NEW`); *Accelerator -> None*, *Internet -> On*; **Save Version -> Save & Run All (Commit)**.

In [ ]:
import glob, os, shutil, subprocess, sys

REPO_URL = "https://github.com/<you>/MeChess.git"      # <- put your repository URL here (not needed when the repository is attached as a dataset)
SIZES = ["5m", "23m", "79m"]
LAMBDAS = "0.1,0.5"                                     # behaviour weights whose adapters are evaluated (fixed BEFORE this run, see the plan)
OUT = "/kaggle/working/humanlike" if os.path.exists("/kaggle") else "humanlike_local"
INPUT_DIR = ""                                          # the folder with the notebook outputs below; empty: search /kaggle/input
# which notebook output (kernel slug) holds what. TRAIN = the test set the adapters were trained on (its judged TRAINING sample gives the tail stand-in), NEW = the confirmation split.
TRAIN = {"testset": "chess-engine-humanlike-testset", "scores": "chess-engine-humanlike-score",
         "judged": {"5m": "chess-engine-humanlike-judge-5m", "23m": "chess-engine-humanlike-judge", "79m": "chess-engine-humanlike-judge-79m"}}
NEW = {"testset": "chess-engine-humanlike-testset2", "scores": "chess-engine-humanlike-score2", "judged": {s: f"chess-engine-humanlike-judge2-{s}" for s in SIZES}}
ADAPTERS = {s: f"chess-engine-humanlike-adapt-{s}-sweep" for s in SIZES}

def sh(*args):
    """Run a command and stream its output; stop the notebook if it fails."""
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"failed: {' '.join(args)}")

if not os.path.exists("chessme"):
    attached = sorted(glob.glob("/kaggle/input/**/chessme/__init__.py", recursive=True), key=lambda f: "/datasets/" not in f)   # prefer the attached repository dataset
    if attached:
        shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), "MeChess")
    else:                                                           # Kaggle: fetch the repository (Internet must be on)
        if "<you>" in REPO_URL:
            raise SystemExit("Attach the repository as a dataset, or set REPO_URL.")
        sh("git", "clone", "--depth", "1", REPO_URL, "MeChess")
    os.chdir("MeChess")
sh(sys.executable, "-m", "pip", "-q", "install", "python-chess", "numpy", "pyyaml", "requests", "zstandard")
CLI = [sys.executable, "-m", "chessme"]              # every step below is one `chessme` command: the same ones you run on a laptop

In [ ]:
root = INPUT_DIR or "/kaggle/input"
def find(slug, name):
    hits = sorted(glob.glob(f"{root}/**/{slug}/**/{name}", recursive=True)) or sorted(glob.glob(f"{root}/**/{slug}/{name}", recursive=True))
    return hits[0] if hits else None
def need(path, what):
    print(f"{what}: {path}")
    if not path:
        raise SystemExit(f"missing input: {what}")
    return path
TB_OLD = need(find(TRAIN["testset"], "testset_TB.jsonl.gz"), "training test set")
TB_NEW = need(find(NEW["testset"], "testset_TB.jsonl.gz"), "confirmation test set")
os.makedirs(OUT, exist_ok=True)

## Step 1. Each size: frozen model and saved adapters on the confirmation split

In [ ]:
for size in SIZES:
    old_scores = need(find(TRAIN["scores"], f"scores_TB_{size}.npz"), f"{size} training scores")
    new_scores = need(find(NEW["scores"], f"scores_TB_{size}.npz"), f"{size} confirmation scores")
    new_judged = need(find(NEW["judged"][size], f"judged_test_{size}.npz"), f"{size} confirmation judged sample")
    jt = find(TRAIN["judged"][size], f"judged_trainval_{size}.npz") or find(TRAIN["judged"][size], "judged_trainval.npz")
    old_judged = need(jt, f"{size} training judged sample")
    adir = os.path.dirname(need(find(ADAPTERS[size], "adapter_all_features_temperature_tail_slot_biases.pt"), f"{size} adapters"))
    sh(*CLI, "humanlike-confirm", "--testset", TB_NEW, "--scores", new_scores, "--judged", new_judged, "--adapters-dir", adir, "--lambdas", LAMBDAS,
       "--train-testset", TB_OLD, "--train-scores", old_scores, "--judged-train", old_judged, "--boot", "1000", "--out", f"{OUT}/confirm_{size}.md")

## Step 2. The gap against model size, on the confirmation split

In [ ]:
MODELS = [f"{s.upper()}={need(find(NEW['scores'], f'scores_TB_{s}.npz'), s + ' scores')},{need(find(NEW['judged'][s], f'judged_test_{s}.npz'), s + ' judged')}" for s in SIZES]
sh(*CLI, "humanlike-scaling", "--testset", TB_NEW, "--models", *MODELS, "--params", *[f"{s.upper()}={s[:-1]}" for s in SIZES], "--boot", "2000", "--out", f"{OUT}/scaling_confirmation.md")